In [1]:
import rasterio
import numpy as np
import pandas as pd

# manually list your 15 TIFF files
tiff_files = [
    "BGD_67_19.tif",
    "BGD_56_68.tif",
    "BGD_40_119.tif",
    "BGD_66_18.tif",
    "BGD_53_80.tif",
    "BGD_59_27.tif",
    "BGD_64_23.tif",
    "BGD_52_81.tif",
    "BGD_62_23.tif",
    "BGD_50_67.tif",
    "BGD_52_67.tif",
    "BGD_54_69.tif",
    "BGD_45_90.tif",
    "BGD_50_83.tif",
    "BGD_43_99.tif"
]

for file_path in tiff_files:
    print(f"\nProcessing: {file_path}")

    with rasterio.open(file_path) as src:
        band = src.read(1)

        flood_points = []
        non_flood_points = []

        height, width = band.shape

        # 🔥 Scan from top-left → row-wise (deterministic)
        for row in range(height):
            for col in range(width):
                val = band[row, col]

                if val == 2 and len(flood_points) < 20000:
                    lon, lat = src.xy(row, col)
                    flood_points.append([lat, lon, 1])  # mapped to 1

                elif val == 1 and len(non_flood_points) < 20000:
                    lon, lat = src.xy(row, col)
                    non_flood_points.append([lat, lon, 0])  # mapped to 0

                # stop early when both done
                if len(flood_points) == 20000 and len(non_flood_points) == 20000:
                    break
            if len(flood_points) == 20000 and len(non_flood_points) == 20000:
                break

        print("Collected Flood:", len(flood_points))
        print("Collected Non-Flood:", len(non_flood_points))

        # safety check
        if len(flood_points) < 20000 or len(non_flood_points) < 20000:
            print("⚠️ Not enough pixels, skipping this file")
            continue

        # combine
        data = flood_points + non_flood_points

    # create dataframe
    df = pd.DataFrame(data, columns=["lat", "lon", "label"])

    # OPTIONAL shuffle (remove if you want strict ordering)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # save CSV with same name
    output_name = file_path.replace(".tif", ".csv")
    df.to_csv(output_name, index=False)

    print(f"✅ Saved: {output_name}")
    print(df["label"].value_counts())


Processing: BGD_67_19.tif
Collected Flood: 20000
Collected Non-Flood: 20000
✅ Saved: BGD_67_19.csv
label
0    20000
1    20000
Name: count, dtype: int64

Processing: BGD_56_68.tif
Collected Flood: 20000
Collected Non-Flood: 20000
✅ Saved: BGD_56_68.csv
label
0    20000
1    20000
Name: count, dtype: int64

Processing: BGD_40_119.tif
Collected Flood: 20000
Collected Non-Flood: 20000
✅ Saved: BGD_40_119.csv
label
0    20000
1    20000
Name: count, dtype: int64

Processing: BGD_66_18.tif
Collected Flood: 20000
Collected Non-Flood: 20000
✅ Saved: BGD_66_18.csv
label
0    20000
1    20000
Name: count, dtype: int64

Processing: BGD_53_80.tif
Collected Flood: 20000
Collected Non-Flood: 20000
✅ Saved: BGD_53_80.csv
label
0    20000
1    20000
Name: count, dtype: int64

Processing: BGD_59_27.tif
Collected Flood: 20000
Collected Non-Flood: 20000
✅ Saved: BGD_59_27.csv
label
0    20000
1    20000
Name: count, dtype: int64

Processing: BGD_64_23.tif
Collected Flood: 20000
Collected Non-Flood: 200

In [1]:
import rasterio
import numpy as np
import pandas as pd
import os

# reproducibility
np.random.seed(42)

# manually list your 15 TIFF files
tiff_files = [
    "BGD_67_19.tif",
    "BGD_56_68.tif",
    "BGD_40_119.tif",
    "BGD_66_18.tif",
    "BGD_53_80.tif",
    "BGD_59_27.tif",
    "BGD_64_23.tif",
    "BGD_52_81.tif",
    "BGD_62_23.tif",
    "BGD_50_67.tif",
    "BGD_52_67.tif",
    "BGD_54_69.tif",
    "BGD_45_90.tif",
    "BGD_50_83.tif",
    "BGD_43_99.tif"
]

for file_path in tiff_files:
    print(f"\n📂 Processing: {file_path}")

    with rasterio.open(file_path) as src:
        band = src.read(1)

        # get indices of flood (2) and non-flood (1)
        flood_idx = np.argwhere(band == 2)
        non_flood_idx = np.argwhere(band == 1)

        print(f"Total Flood Pixels: {len(flood_idx)}")
        print(f"Total Non-Flood Pixels: {len(non_flood_idx)}")

        # shuffle indices
        np.random.shuffle(flood_idx)
        np.random.shuffle(non_flood_idx)

        # choose sample size (max 20k each but safe)
        max_samples = 20000
        n_flood = min(max_samples, len(flood_idx))
        n_non_flood = min(max_samples, len(non_flood_idx))

        if n_flood == 0 or n_non_flood == 0:
            print("⚠️ Skipping (no valid pixels)")
            continue

        flood_points = []
        non_flood_points = []

        # collect flood points
        for row, col in flood_idx[:n_flood]:
            lon, lat = src.xy(row, col)
            flood_points.append([lat, lon, 1])

        # collect non-flood points
        for row, col in non_flood_idx[:n_non_flood]:
            lon, lat = src.xy(row, col)
            non_flood_points.append([lat, lon, 0])

        print(f"✅ Sampled Flood: {len(flood_points)}")
        print(f"✅ Sampled Non-Flood: {len(non_flood_points)}")

        # combine
        data = flood_points + non_flood_points

    # create dataframe
    df = pd.DataFrame(data, columns=["lat", "lon", "label"])

    # final shuffle (optional but good)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # 🔥 improved file naming
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    output_name = f"{base_name}_balanced_random_40k.csv"

    df.to_csv(output_name, index=False)

    print(f"💾 Saved: {output_name}")
    print(df["label"].value_counts())


📂 Processing: BGD_67_19.tif
Total Flood Pixels: 461744
Total Non-Flood Pixels: 586832
✅ Sampled Flood: 20000
✅ Sampled Non-Flood: 20000
💾 Saved: BGD_67_19_balanced_random_40k.csv
label
0    20000
1    20000
Name: count, dtype: int64

📂 Processing: BGD_56_68.tif
Total Flood Pixels: 610535
Total Non-Flood Pixels: 438041
✅ Sampled Flood: 20000
✅ Sampled Non-Flood: 20000
💾 Saved: BGD_56_68_balanced_random_40k.csv
label
0    20000
1    20000
Name: count, dtype: int64

📂 Processing: BGD_40_119.tif
Total Flood Pixels: 155865
Total Non-Flood Pixels: 892711
✅ Sampled Flood: 20000
✅ Sampled Non-Flood: 20000
💾 Saved: BGD_40_119_balanced_random_40k.csv
label
0    20000
1    20000
Name: count, dtype: int64

📂 Processing: BGD_66_18.tif
Total Flood Pixels: 243393
Total Non-Flood Pixels: 805183
✅ Sampled Flood: 20000
✅ Sampled Non-Flood: 20000
💾 Saved: BGD_66_18_balanced_random_40k.csv
label
0    20000
1    20000
Name: count, dtype: int64

📂 Processing: BGD_53_80.tif
Total Flood Pixels: 221665
Total 

In [2]:
import pandas as pd
import glob

files = glob.glob("*_balanced_random_40k.csv")

total_rows = 0

for f in files:
    df = pd.read_csv(f)
    print(f"{f}: {len(df)} rows")
    total_rows += len(df)

print("\n🔥 Total data points:", total_rows)

BGD_64_23_balanced_random_40k.csv: 40000 rows
BGD_56_68_balanced_random_40k.csv: 40000 rows
BGD_45_90_balanced_random_40k.csv: 40000 rows
BGD_50_67_balanced_random_40k.csv: 40000 rows
BGD_66_18_balanced_random_40k.csv: 40000 rows
BGD_62_23_balanced_random_40k.csv: 40000 rows
BGD_54_69_balanced_random_40k.csv: 40000 rows
BGD_53_80_balanced_random_40k.csv: 40000 rows
BGD_67_19_balanced_random_40k.csv: 40000 rows
BGD_52_67_balanced_random_40k.csv: 40000 rows
BGD_43_99_balanced_random_40k.csv: 40000 rows
BGD_50_83_balanced_random_40k.csv: 40000 rows
BGD_52_81_balanced_random_40k.csv: 40000 rows
BGD_40_119_balanced_random_40k.csv: 40000 rows
BGD_59_27_balanced_random_40k.csv: 40000 rows

🔥 Total data points: 600000


In [3]:
total_flood = 0
total_non_flood = 0

for f in files:
    df = pd.read_csv(f)
    total_flood += (df["label"] == 1).sum()
    total_non_flood += (df["label"] == 0).sum()

print("\nFlood:", total_flood)
print("Non-Flood:", total_non_flood)
print("Total:", total_flood + total_non_flood)


Flood: 300000
Non-Flood: 300000
Total: 600000


In [4]:
import rasterio
import numpy as np

tiff_files = [
    "BGD_67_19.tif", "BGD_56_68.tif", "BGD_40_119.tif",
    "BGD_66_18.tif", "BGD_53_80.tif", "BGD_59_27.tif",
    "BGD_64_23.tif", "BGD_52_81.tif", "BGD_62_23.tif",
    "BGD_50_67.tif", "BGD_52_67.tif", "BGD_54_69.tif",
    "BGD_45_90.tif", "BGD_50_83.tif", "BGD_43_99.tif"
]

total_pixels_all = 0
total_flood_all = 0
total_non_flood_all = 0

print("📊 Per TIFF Stats:\n")

for file_path in tiff_files:
    with rasterio.open(file_path) as src:
        band = src.read(1)

        total_pixels = band.size
        flood_pixels = np.sum(band == 2)
        non_flood_pixels = np.sum(band == 1)

        total_pixels_all += total_pixels
        total_flood_all += flood_pixels
        total_non_flood_all += non_flood_pixels

        print(f"{file_path}")
        print(f"  Total Pixels     : {total_pixels}")
        print(f"  Flood Pixels     : {flood_pixels}")
        print(f"  Non-Flood Pixels : {non_flood_pixels}")
        print("-" * 40)

print("\n🔥 Overall Stats:")
print(f"Total Pixels     : {total_pixels_all}")
print(f"Total Flood      : {total_flood_all}")
print(f"Total Non-Flood  : {total_non_flood_all}")

📊 Per TIFF Stats:

BGD_67_19.tif
  Total Pixels     : 1048576
  Flood Pixels     : 461744
  Non-Flood Pixels : 586832
----------------------------------------
BGD_56_68.tif
  Total Pixels     : 1048576
  Flood Pixels     : 610535
  Non-Flood Pixels : 438041
----------------------------------------
BGD_40_119.tif
  Total Pixels     : 1048576
  Flood Pixels     : 155865
  Non-Flood Pixels : 892711
----------------------------------------
BGD_66_18.tif
  Total Pixels     : 1048576
  Flood Pixels     : 243393
  Non-Flood Pixels : 805183
----------------------------------------
BGD_53_80.tif
  Total Pixels     : 1048576
  Flood Pixels     : 221665
  Non-Flood Pixels : 826911
----------------------------------------
BGD_59_27.tif
  Total Pixels     : 1048576
  Flood Pixels     : 358033
  Non-Flood Pixels : 690543
----------------------------------------
BGD_64_23.tif
  Total Pixels     : 1048576
  Flood Pixels     : 382302
  Non-Flood Pixels : 666274
----------------------------------------
